In [1]:
# Install required libraries silently
!pip install chromadb rank_bm25 sentence-transformers underthesea datasets -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 63.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 80.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 61.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 64.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━

In [2]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import json
import pickle
import math
import shutil
import torch
import numpy as np
import torch.nn.functional as F
import chromadb

from tqdm.auto import tqdm
from datasets import load_dataset
from underthesea import word_tokenize
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import CrossEncoder

# ── Configuration (Mapped from config.py) ────────────────────────────────────
DATASET_NAME = "taidng/UIT-ViQuAD2.0"
EVAL_SPLIT = "validation"

# Paths
INDICES_DIR = "/kaggle/input/datasets/khanhtamnd/viet-rag-indices" 
SOURCE_CHROMA = os.path.join(INDICES_DIR, "chroma_db")
BM25_PATH = os.path.join(INDICES_DIR, "bm25_index.pkl")
CHROMA_DIR = "/kaggle/working/chroma_db"

# Copy ChromaDB to a writable directory in Kaggle
if not os.path.exists(CHROMA_DIR):
    print(f"Copying ChromaDB to writable directory: {CHROMA_DIR}...")
    shutil.copytree(SOURCE_CHROMA, CHROMA_DIR)

CHROMA_COLLECTION = "virag_chunks"

# Models
EMBEDDING_MODEL = "Alibaba-NLP/gte-multilingual-base"
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Retrieval Hyperparameters
DENSE_TOP_K = 100
BM25_TOP_K = 100
RRF_K = 60
HYBRID_TOP_K = 80
RERANK_TOP_K = 10

MAX_EVAL_SAMPLES = None  # Set to None for the full dataset

Copying ChromaDB to writable directory: /kaggle/working/chroma_db...


In [3]:
class KaggleHybridRetriever:
    def __init__(self):
        print(f"[Retriever] Loading embedding model on {DEVICE.upper()} (Raw Transformers) …")

        self.tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL, trust_remote_code=True)
        self.embed_model = AutoModel.from_pretrained(
            EMBEDDING_MODEL,
            trust_remote_code=True,
            ignore_mismatched_sizes=True,
        ).to(DEVICE)
        self.embed_model.eval()

        print(f"[Retriever] Connecting to ChromaDB at {CHROMA_DIR} …")
        self.chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)
        self.collection = self.chroma_client.get_collection(CHROMA_COLLECTION)

        print(f"[Retriever] Loading BM25 index from {BM25_PATH} …")
        with open(BM25_PATH, "rb") as f:
            bm25_payload = pickle.load(f)

        self.bm25 = bm25_payload["bm25"]
        self.bm25_chunk_ids = bm25_payload["chunk_ids"]
        self.bm25_texts = bm25_payload["texts"]

        print(f"[Retriever] Loading CrossEncoder reranker on {DEVICE.upper()} …")
        self.reranker = CrossEncoder(
            RERANKER_MODEL,
            device=DEVICE,
            max_length=512
        )

        print("[Retriever] Ready ✓")

    def _encode_query(self, query: str):
        encoded = self.tokenizer(
            [query],
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        ).to(DEVICE)
        
        seq_len = encoded["input_ids"].shape[1]
        encoded["position_ids"] = torch.arange(seq_len, dtype=torch.long, device=DEVICE).unsqueeze(0).expand(1, -1)
        
        with torch.no_grad():
            out = self.embed_model(**encoded, unpad_inputs=False)  
            
        hidden_states = torch.nan_to_num(out.last_hidden_state, nan=0.0)
        
        mask = encoded["attention_mask"].unsqueeze(-1).expand(hidden_states.size()).float()
        sum_embeddings = torch.sum(hidden_states * mask, 1)
        sum_mask = torch.clamp(mask.sum(1), min=1e-9)
        emb = sum_embeddings / sum_mask
        
        emb = F.normalize(emb, p=2, dim=1)
        return emb[0].cpu().float().tolist()

    def _dense_search(self, query: str):
        # FIX: Removed the "query: " prefix to perfectly match the build index
        query_emb = self._encode_query(query)
    
        results = self.collection.query(
            query_embeddings=[query_emb],
            n_results=DENSE_TOP_K,
            include=["documents"]
        )
    
        return [
            {
                "chunk_id": doc_id,
                "text": results["documents"][0][i],
                "dense_rank": i + 1,
            }
            for i, doc_id in enumerate(results["ids"][0])
        ]

    def _bm25_search(self, query: str):
        # FIX: Use underthesea format="text" to preserve compound word underscores
        tokenised_query = word_tokenize(query.lower(), format="text").split()
        
        scores = self.bm25.get_scores(tokenised_query)
        top_indices = np.argsort(scores)[::-1][:BM25_TOP_K]
        
        hits = []
        for rank, idx in enumerate(top_indices):
            hits.append({
                "chunk_id": self.bm25_chunk_ids[idx],
                "text": self.bm25_texts[idx],
                "bm25_rank": rank + 1
            })
        return hits

    def retrieve(self, query: str):
        dense_hits = self._dense_search(query)
        bm25_hits = self._bm25_search(query)

        scores, doc_store = {}, {}

        for hit in dense_hits:
            cid = hit["chunk_id"]
            scores[cid] = scores.get(cid, 0.0) + 1.0 / (RRF_K + hit["dense_rank"])
            doc_store[cid] = hit

        for hit in bm25_hits:
            cid = hit["chunk_id"]
            scores[cid] = scores.get(cid, 0.0) + 1.0 / (RRF_K + hit["bm25_rank"])
            doc_store.setdefault(cid, hit)

        fused = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:HYBRID_TOP_K]
        candidates = [doc_store[cid] for cid, _ in fused]

        safe_pairs, safe_candidates = [], []
        for c in candidates:
            text = c.get("text")
            if not isinstance(text, str): continue
            
            token_len = len(text.split())
            if token_len > 600: continue 

            safe_pairs.append((query, text))
            safe_candidates.append(c)

        if not safe_pairs:
            return []

        rerank_scores = self.reranker.predict(safe_pairs, batch_size=8, show_progress_bar=False)

        for cand, score in zip(safe_candidates, rerank_scores):
            cand["rerank_score"] = float(score)

        return sorted(safe_candidates, key=lambda x: x["rerank_score"], reverse=True)[:RERANK_TOP_K]

In [4]:
def is_relevant(text: str, gt: str, threshold: float = 0.6) -> bool:
    text, gt = text.strip(), gt.strip()
    if not text: return False
    
    window = 40
    if len(text) < window: return text in gt
    
    hits = sum(1 for i in range(0, len(text) - window, window // 2) if text[i:i+window] in gt)
    total_windows = max(1, len(range(0, len(text) - window, window // 2)))
    return (hits / total_windows) >= threshold

def recall_at_k(retrieved_ids: list[str], relevant_ids: set[str], k: int) -> float:
    return float(any(rid in relevant_ids for rid in retrieved_ids[:k]))

def ndcg_at_k(retrieved_ids: list[str], relevant_ids: set[str], k: int) -> float:
    # Calculate actual DCG
    dcg = sum(1.0 / math.log2(rank + 1) for rank, rid in enumerate(retrieved_ids[:k], 1) if rid in relevant_ids)
    
    # Calculate Ideal DCG based on how many relevant chunks actually exist
    num_relevant = min(len(relevant_ids), k)
    idcg = sum(1.0 / math.log2(rank + 1) for rank in range(1, num_relevant + 1))
    
    return dcg / idcg if idcg > 0 else 0.0

# ── 1. Init Retriever & Dataset ──
retriever = KaggleHybridRetriever()

print(f"\nLoading dataset: {DATASET_NAME} / {EVAL_SPLIT} …")
ds = load_dataset(DATASET_NAME)[EVAL_SPLIT]

# ---> ADD THIS FILTERING BLOCK <---
print(f"Original dataset size: {len(ds)}")
# Filter out rows where the "text" list inside "answers" is empty
ds = ds.filter(lambda row: len(row["answers"]["text"]) > 0)
print(f"Filtered dataset size (answerable only): {len(ds)}")
# ----------------------------------

if MAX_EVAL_SAMPLES:
    # Safely select samples (ensure we don't request more than available)
    actual_samples = min(MAX_EVAL_SAMPLES, len(ds))
    ds = ds.select(range(actual_samples))

# ── 2. Hybrid Evaluation Loop ──
recall_scores, ndcg_scores, failures = [], [], []

for row in tqdm(ds, desc="Evaluating Queries"):
    query, gt_context = row["question"], row["context"].strip()
    
    # Extract short answers if they exist
    gold_answers = [ans.lower().strip() for ans in row["answers"]["text"]]
    
    passages = retriever.retrieve(query)
    retrieved_texts = [p["text"] for p in passages]
    
    relevant_mask = []
    for text in retrieved_texts:
        # Check 1: Does it contain any of the exact answer strings? (Handles answerable queries)
        if gold_answers and any(ans in text.lower() for ans in gold_answers):
            relevant_mask.append(True)
        # Check 2: Fallback to sliding window paragraph overlap (Handles unanswerable queries cleanly)
        elif is_relevant(text, gt_context):
            relevant_mask.append(True)
        else:
            relevant_mask.append(False)
            
    retrieved_ids = [str(i) for i in range(len(retrieved_texts))]
    relevant_ids = {str(i) for i, m in enumerate(relevant_mask) if m}
    
    recall_scores.append(recall_at_k(retrieved_ids, relevant_ids, RERANK_TOP_K))
    ndcg_scores.append(ndcg_at_k(retrieved_ids, relevant_ids, RERANK_TOP_K))
    
    # Error analysis saves true failures
    if not any(relevant_mask):
        failures.append({
            "question": query,
            "gold_answers": gold_answers,
            "ground_truth_context": gt_context,
            "top_3_retrieved": [f"[Score: {p.get('rerank_score', 0):.2f}] {p['text']}" for p in passages[:3]]
        })

print(f"\n{'='*40}")
print(f"Evaluation Results (n={len(recall_scores)})")
print(f"{'='*40}")
print(f"Recall@{RERANK_TOP_K}  : {np.mean(recall_scores):.4f}  (target ≥ 0.9000)")
print(f"nDCG@{RERANK_TOP_K}    : {np.mean(ndcg_scores):.4f}  (target ≥ 0.8920)")

with open("/kaggle/working/failures.json", "w", encoding="utf-8") as f:
    json.dump(failures, f, ensure_ascii=False, indent=4)
print(f"\nSaved {len(failures)} true failures to /kaggle/working/failures.json")

[Retriever] Loading embedding model on CUDA (Raw Transformers) …


config.json: 0.00B [00:00, ?B/s]

configuration.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/Alibaba-NLP/new-impl:
- configuration.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Original dataset size: 3814


Filter:   0%|          | 0/3814 [00:00<?, ? examples/s]

Filtered dataset size (answerable only): 2653


Evaluating Queries:   0%|          | 0/2653 [00:00<?, ?it/s]


Evaluation Results (n=2653)
Recall@10  : 0.9175  (target ≥ 0.9000)
nDCG@10    : 0.8632  (target ≥ 0.8920)

Saved 219 true failures to /kaggle/working/failures.json
